# 2022_12_12

## pattern to occur

기존의 방식은 패턴 자체를 학습후 이를 이용해서 사건의 발생을 판단함  
ewma 패턴에 의존적이고, TFT는 사건 이후의 패턴에 대해서만 예측을 함  
이를 개선하기 위해서  


* ewma를 1시간 정도를 고정(사건의 발생 여부 파악을 위해서) 
* 학습(hparams를 small, middle, large)  
* valdation을 하기 위해서 데이터 파일을 분리

### ewma 
1. ewma를 가장 작게 적용후 noise 추가
2. 사건 발생 할때의 index를 가져와서 indedx-1, index+1에는 0.5씩 추가해줌


``` python
def moving_average_alpha_noise(df: pd.DataFrame, unit: float):       
    for dong in df['h_dong'].unique():
        dong_df = df[df['h_dong'] == dong]
        max_value = dong_df['count'].max()
        back_ewma = dong_df['count'].ewm(alpha = unit).mean()

        inv_dong_df = dong_df[::-1]
        for_ewma = inv_dong_df['count'].ewm(alpha = unit).mean()
        
        ewma = (for_ewma + back_ewma) / 2
        ewma = ewma / ewma.max() * max_value
        df['count'][dong_df.index] = ewma 
    df['count'] += np.random.normal(0.02,0.04, len(df))
    return df

def moving_average_hour(df: pd.DataFrame, unit = 0):
    for dong in df['h_dong'].unique():
        #print(dong)
        dong_df = df[df['h_dong'] == dong]
        over0 = dong_df[dong_df['count'] > 0].index
        for idx in over0:
            #print(idx , df['time_idx'].loc[idx])
            value = df['count'].loc[idx] 
            try:
                df['count'].loc[idx-21] += value / 2
                df['count'].loc[idx+21] += value / 2
            except:
                pass
    return df

```

> code test notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_09_TESt_pattern2occur/ewma_to_scaleup.ipynb

### train 
> train notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_09_TESt_pattern2occur/training_GPU1_all.ipynb

### test data split 
> split notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_09_TESt_pattern2occur/test_data_split.ipynb


### result 
> result notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2022_12_09_TESt_pattern2occur/val_test.ipynb

* noise 추가 : 에러남
```
ValueError: 672 (19.05%) of count values were found to be NA or infinite (even after encoding). NA values are not allowed `allow_missing_timesteps` refers to missing rows, not to missing values. Possible strategies to fix the issue are (a) dropping the variable count, (b) using `NaNLabelEncoder(add_nan=True)` for categorical variables, (c) filling missing values and/or (d) optionally adding a variable indicating filled values
```

* hour processing : prediction이 모두 0임
![](https://i.imgur.com/ZzGSgqz.png)
